# CD-HIT 蛋白序列关联样本元数据

在项目根目录运行本 Notebook。以 CD-HIT 的代表 FASTA 为输入，每条蛋白输出一行。
如果当前内核缺少 openpyxl，在代码单元中运行 %pip install openpyxl 后再执行本流程。

- **fasta_sample_id** 来自蛋白 ID，去掉末尾的七位基因编号。
- **s1_sample_id** 和 **s1_accession** 分别是补充表 S1 的样本编号和数据登录号。
- **s1_row** 是样本在原工作表中的行号，便于回查。
- 温度字段保留原表中的 N/A。

In [1]:
import csv
from itertools import zip_longest
import os
from pathlib import Path
import re
from tempfile import NamedTemporaryFile

from openpyxl import load_workbook

project = Path.cwd()
fasta_path = project / 'output/cdhit/meer1000M_cd90.pep.fa'
s1_path = project / 'dataset/MEER/DSGC_Table_S1.xlsx'
alias_path = project / 'dataset/论文附录数据/07_DSGC_CHM2026_TableS1_metagenome_metadata_latlon.xlsx'
output_path = fasta_path.with_name('meer1000M_cd90_DSGC_Table_S1.csv')

## 1. 建立样本对应关系

优先匹配原样本 ID，其次统一下划线与连字符并检查对应关系唯一。
FDZ064-BuG10-12-7 根据两版 S1 第 420 行对应到 FDZ064-BuG10-12。
两行的经纬度、深度、材料和总测序量一致。导出的登录号和元数据取自 DSGC_Table_S1。

In [2]:
def read_s1(path):
    workbook = load_workbook(path, read_only=True, data_only=True)
    try:
        rows = workbook['S1'].iter_rows(min_row=4, values_only=True)
        return {n: row for n, row in enumerate(rows, start=4) if row[1]}
    finally:
        workbook.close()


def sample_fields(cell):
    stem, separator, accession = cell.strip().rpartition(' (')
    if separator and accession.endswith(')'):
        return stem.strip(), accession[:-1]
    return cell.strip(), ''


def normalize_sample_id(sample_id):
    return re.sub(r'[-_]+', '-', sample_id).strip('-')


s1_rows = read_s1(s1_path)
samples = {}
normalized_samples = {}
for row_number, row in s1_rows.items():
    sample_id, accession = sample_fields(row[1])
    normalized = normalize_sample_id(sample_id)
    if sample_id in samples or normalized in normalized_samples:
        raise ValueError(f'S1 样本编号不唯一：{sample_id}')
    samples[sample_id] = (row_number, accession, row)
    normalized_samples[normalized] = sample_id

original_row = s1_rows[420]
alias_row = read_s1(alias_path)[420]
original_id, _ = sample_fields(original_row[1])
alias_id, _ = sample_fields(alias_row[1])
if (original_id, alias_id) != ('FDZ064-BuG10-12', 'FDZ064-BuG10-12-7'):
    raise ValueError('两版 S1 第 420 行的样本编号有变动')
# 两版 S1 使用不同登录号。核对这两列之外的样本元数据。
for i, (original, revised) in enumerate(zip(original_row, alias_row)):
    if i not in (1, 9) and original != revised:
        raise ValueError('两版 S1 第 420 行的样本元数据不一致')


def lookup_sample(sample_id):
    if sample_id in samples:
        return sample_id
    canonical = normalized_samples.get(normalize_sample_id(sample_id))
    if canonical is not None:
        return canonical
    if sample_id == alias_id:
        return original_id
    raise ValueError(f'找不到样本元数据：{sample_id}')

## 2. 拼接代表 FASTA 与元数据

按 FASTA 原顺序输出，每条氨基酸序列拼成一列。

In [3]:
columns = [
    'protein_id', 'fasta_sample_id', 's1_sample_id', 's1_accession',
    's1_row', 'aa_length', 'sequence', 'fasta_description',
    'ecosystem_group', 'ncbi_taxonomy',
    'isolation_source_environmental_context', 'sample_materials',
    'latitude', 'longitude', 'depth_m', 'temperature_c',
    'related_srr_ids', 'total_bases_gb', 'source_doi_and_reference',
]


def fasta_records(path):
    with path.open(encoding='utf-8') as handle:
        header = None
        parts = []
        for raw_line in handle:
            line = raw_line.strip()
            if line.startswith('>'):
                if header is not None:
                    yield header, ''.join(parts)
                header = line[1:]
                parts = []
            elif line:
                parts.append(line)
        if header is not None:
            yield header, ''.join(parts)


def joined_rows():
    for header, sequence in fasta_records(fasta_path):
        protein_id, description = header.split(maxsplit=1)
        sample_id, _, gene_number = protein_id.rpartition('_')
        if not sample_id or not re.fullmatch(r'[0-9]{7}', gene_number):
            raise ValueError(f'蛋白 ID 格式异常：{protein_id}')
        if not sequence:
            raise ValueError(f'序列为空：{protein_id}')

        canonical = lookup_sample(sample_id)
        row_number, accession, metadata = samples[canonical]
        yield (
            protein_id, sample_id, canonical, accession, row_number,
            len(sequence), sequence, description, metadata[0], *metadata[2:],
        )

## 3. 导出并核验 CSV

逐行核对拼接结果。重复运行时直接核验已有 CSV。
更换输入数据后，可以调整 output_path 导出到新的文件。

In [4]:
def verify_csv(path):
    protein_ids = set()
    sample_ids = set()
    with path.open(encoding='utf-8-sig', newline='') as handle:
        reader = csv.reader(handle, strict=True)
        if next(reader, None) != columns:
            raise ValueError(f'CSV 列结构不同，请检查输出路径：{path.name}')

        for line_number, (saved, expected) in enumerate(
            zip_longest(reader, joined_rows()), start=2
        ):
            if saved is None or expected is None:
                raise ValueError(f'CSV 行数与 FASTA 不一致：第 {line_number} 行')
            expected_text = [str(value) if value is not None else '' for value in expected]
            if saved != expected_text:
                raise ValueError(f'CSV 内容与输入不一致：第 {line_number} 行')
            protein_id, _, canonical_id = saved[:3]
            if protein_id in protein_ids:
                raise ValueError(f'蛋白 ID 重复：{protein_id}')
            protein_ids.add(protein_id)
            sample_ids.add(canonical_id)

    if not protein_ids:
        raise ValueError('输入 FASTA 没有序列')
    return len(protein_ids), len(sample_ids)


def export_csv(path):
    if path.exists():
        return verify_csv(path)

    temporary_path = None
    try:
        with NamedTemporaryFile(
            mode='w', dir=path.parent, prefix=f'.{path.stem}_', suffix='.csv',
            encoding='utf-8-sig', newline='', delete=False,
        ) as handle:
            temporary_path = Path(handle.name)
            writer = csv.writer(handle)
            writer.writerow(columns)
            writer.writerows(joined_rows())

        counts = verify_csv(temporary_path)
        os.link(temporary_path, path)
        return counts
    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)


protein_count, sample_count = export_csv(output_path)
print(f'{output_path.name} | {protein_count:,} 条蛋白序列 | {sample_count:,} 个样本')

meer1000M_cd90_DSGC_Table_S1.csv | 386,829 条蛋白序列 | 1,121 个样本


## 4. 配置 Swiss-Prot 蛋白比对

用 CD-HIT 代表序列 FASTA（`fasta_path`）查询本地 UniProtKB/Swiss-Prot，保留原始蛋白 ID（`qseqid`），可与前面 CSV 的 `protein_id` 关联。默认使用 12 个线程和 `--sensitive`，保留每条查询最多 10 个 E-value ≤ 1e-5 的目标。以下代码记录输入文件的 SHA-256、DIAMOND 版本和参数，便于重跑同一分析。

[DIAMOND 参数文档](https://github.com/bbuchfink/diamond/wiki/3.-Command-line-options)

In [5]:
from contextlib import suppress
from hashlib import sha256
import shlex
import shutil
import subprocess
from tempfile import TemporaryDirectory

swissprot_fasta = project / 'dataset/protein_annotation/uniprot_sprot.fasta'
diamond_dir = project / 'output/diamond'
threads = 12
sensitivity = '--sensitive'
evalue = 1e-5
max_targets = 10

bundled_diamond = project / 'dataset/MEER/MEER/annotation_2026-11-05/tools/diamond'
diamond = str(bundled_diamond) if os.access(bundled_diamond, os.X_OK) else shutil.which('diamond')
if diamond is None:
    raise FileNotFoundError('找不到 DIAMOND。请在当前环境安装 diamond，或设置 diamond 可执行文件路径。')
diamond_version = subprocess.check_output([diamond, 'version'], text=True).strip()

hit_columns = [
    'qseqid', 'sseqid', 'pident', 'length', 'qlen', 'slen',
    'qstart', 'qend', 'sstart', 'send', 'qcovhsp', 'scovhsp',
    'evalue', 'bitscore', 'stitle',
]
search_args = [
    '--threads', str(threads), sensitivity, '--evalue', str(evalue),
    '--max-target-seqs', str(max_targets), '--outfmt', '6', *hit_columns,
]


def file_sha256(path):
    digest = sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


query_sha = file_sha256(fasta_path)
reference_sha = file_sha256(swissprot_fasta)
db_key = sha256(repr((diamond_version, reference_sha)).encode()).hexdigest()[:12]
run_key = sha256(repr((db_key, query_sha, search_args)).encode()).hexdigest()[:12]
db_path = diamond_dir / f'uniprot_sprot_{db_key}.dmnd'
hits_path = diamond_dir / f'meer1000M_cd90_vs_swissprot_{run_key}.tsv'

print(diamond_version)
print(f'查询 FASTA  {fasta_path.relative_to(project)}\nSHA-256    {query_sha}')
print(f'参考 FASTA  {swissprot_fasta.relative_to(project)}\nSHA-256    {reference_sha}')

diamond version 2.2.8
查询 FASTA  output/cdhit/meer1000M_cd90.pep.fa
SHA-256    0858ce53cedeccc9f70a97093ebee6cbf69f93e1ff13d2440e2dd237b564ae90
参考 FASTA  dataset/protein_annotation/uniprot_sprot.fasta
SHA-256    1f0a0d0a08d4c932ce91dca0b614fadfca87bd00931d7aad747c523f604d4f46


## 5. 建库并运行 DIAMOND blastp

在 `output/diamond/` 保存数据库和带表头的命中表。文件名随输入或相应配置变化。同一配置的重复运行复用已经完整写出的文件。

In [6]:
def run_diamond(command, stdout=None):
    print(shlex.join(command), flush=True)
    result = subprocess.run(command, stdout=stdout, stderr=subprocess.PIPE, text=True)
    print('\n'.join(result.stderr.splitlines()[-8:]))
    result.check_returncode()


diamond_dir.mkdir(parents=True, exist_ok=True)

makedb_args = [diamond, 'makedb', '--in', str(swissprot_fasta), '--threads', str(threads)]
blastp_args = [diamond, 'blastp', '--query', str(fasta_path), '--db', str(db_path), *search_args]

if not db_path.exists() or not hits_path.exists():
    with TemporaryDirectory(dir=diamond_dir, prefix='.diamond_') as work_dir:
        work_dir = Path(work_dir)
        if not db_path.exists():
            temp_db = work_dir / db_path.name
            makedb_cmd = [*makedb_args, '--db', str(temp_db)]
            run_diamond(makedb_cmd)
            with suppress(FileExistsError):
                os.link(temp_db, db_path)

        if not hits_path.exists():
            temp_hits = work_dir / hits_path.name
            blastp_cmd = [*blastp_args, '--tmpdir', str(work_dir)]
            # DIAMOND 的命中记录写到标准输出，直接接在表头后。
            with temp_hits.open('wb') as handle:
                handle.write(('\t'.join(hit_columns) + '\n').encode('utf-8'))
                handle.flush()
                run_diamond(blastp_cmd, stdout=handle)
            with suppress(FileExistsError):
                os.link(temp_hits, hits_path)
else:
    print('复用已有数据库和命中表。')

print(f'命中表  {hits_path.relative_to(project)}')

/Users/conwaysu/Documents/psychrophilic/Psychrophilic-protein-discovery/dataset/MEER/MEER/annotation_2026-11-05/tools/diamond makedb --in /Users/conwaysu/Documents/psychrophilic/Psychrophilic-protein-discovery/dataset/protein_annotation/uniprot_sprot.fasta --threads 12 --db /Users/conwaysu/Documents/psychrophilic/Psychrophilic-protein-discovery/output/diamond/.diamond_lzw1onld/uniprot_sprot_8e1f2b65e49c.dmnd
Database sequences  576073
  Database letters  209202719
     Database hash  77f6fc48d3f7dab51dba9b5dfb13c9d3
        Total time  0.803000s

/Users/conwaysu/Documents/psychrophilic/Psychrophilic-protein-discovery/dataset/MEER/MEER/annotation_2026-11-05/tools/diamond blastp --query /Users/conwaysu/Documents/psychrophilic/Psychrophilic-protein-discovery/output/cdhit/meer1000M_cd90.pep.fa --db /Users/conwaysu/Documents/psychrophilic/Psychrophilic-protein-discovery/output/diamond/uniprot_sprot_8e1f2b65e49c.dmnd --threads 12 --sensitive --evalue 1e-05 --max-target-seqs 10 --outfmt 6 qse

## 6. 核对命中数并预览结果

`sseqid` 含 Swiss-Prot accession（`sp|accession|entry` 的中间字段）。`pident` 是局部比对的一致性百分比，`qcovhsp` 和 `scovhsp` 分别表示该局部比对对查询和目标蛋白的覆盖百分比。解读命中时同时查看 E-value、一致性和覆盖度。相似性结果可用于提出功能或嗜冷性假设，后续仍需验证。

In [7]:
from IPython.display import display
import pandas as pd

matched_queries = set()
hit_count = 0
preview = []
with hits_path.open(encoding='utf-8', newline='') as handle:
    reader = csv.reader(handle, delimiter='\t', quoting=csv.QUOTE_NONE)
    if next(reader, None) != hit_columns:
        raise ValueError(f'命中表的列名与配置不符：{hits_path}')
    for row in reader:
        if len(row) != len(hit_columns):
            raise ValueError(f'命中表第 {reader.line_num} 行列数异常')
        matched_queries.add(row[0])
        hit_count += 1
        if len(preview) < 5:
            preview.append(row)

matched_count = len(matched_queries)
if matched_count > protein_count or hit_count > matched_count * max_targets:
    raise ValueError('命中计数与查询序列数或每条查询的目标数上限不一致')
print(f'有命中的查询  {matched_count:,} / {protein_count:,}（{matched_count / protein_count:.1%}）')
print(f'无命中的查询  {protein_count - matched_count:,}')
print(f'命中记录数    {hit_count:,}')

preview_columns = ['qseqid', 'sseqid', 'pident', 'qcovhsp', 'scovhsp', 'evalue', 'bitscore']
if preview:
    display(pd.DataFrame(preview, columns=hit_columns)[preview_columns])

有命中的查询  238,478 / 386,829（61.6%）
无命中的查询  148,351
命中记录数    1,997,487


,qseqid,sseqid,pident,qcovhsp,scovhsp,evalue,bitscore
0,FDZ031YE0-5_0000002,sp|P45394|YRBG_ECOLI,33.3,89.2,83.7,7.47e-48,166
1,FDZ031YE0-5_0000002,sp|Q0ZAI3|CAXA_ALKAM,36.8,82.4,78.1,1.50e-45,159
2,FDZ031YE0-5_0000002,sp|Q8TPA6|MAX1_METAC,29.9,96.1,92.6,3.76e-36,135
3,FDZ031YE0-5_0000002,sp|Q57556|Y091_METJA,35.6,81.4,78.1,5.14e-36,134
4,FDZ031YE0-5_0000003,sp|Q82C66|SYL_STRAW,56.4,100,66.8,5.55e-246,708


## 7. 获取 UniProt 温度依赖注释

从 UniProt 官方 API 获取当前 Swiss-Prot 的 `accession` 与 `temp_dependence` 字段，按命中登录号关联。下载完整的已审阅条目列表，以区分“条目存在但字段为空”和“登录号未在当前列表中找到”。网络错误、响应不完整或未找到登录号会中止导出。

按 UTC 日期保存完整查询快照，重复运行复用当日快照。快照包含查询网址、UniProt 版本、获取时间及原始响应 SHA-256。温度注释保留原文及证据编号。

[UniProt API 文档](https://www.uniprot.org/help/api_queries)

In [8]:
from datetime import datetime, timezone
from http.client import IncompleteRead
import io
import json
import time
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

temperature_url = 'https://rest.uniprot.org/uniprotkb/stream?' + urlencode({
    'query': 'reviewed:true',
    'format': 'tsv',
    'fields': 'accession,temp_dependence',
})
snapshot_day = datetime.now(timezone.utc).strftime('%Y%m%d')
temperature_cache = diamond_dir / f'uniprot_swissprot_temperature_{snapshot_day}.json'

if not temperature_cache.exists():
    for attempt in range(4):
        try:
            request = Request(temperature_url, headers={'User-Agent': 'SwissProt-temperature-notebook/1.0'})
            with urlopen(request, timeout=60) as response:
                payload = response.read()
                release = response.headers.get('X-UniProt-Release')
                total_header = response.headers.get('X-Total-Results')
            break
        except (URLError, TimeoutError, IncompleteRead) as error:
            if isinstance(error, HTTPError) and error.code not in (429, 500, 502, 503, 504):
                raise
            if attempt == 3:
                raise
            time.sleep(2 ** attempt)

    rows = csv.reader(io.StringIO(payload.decode('utf-8')), delimiter='\t')
    if next(rows, None) != ['Entry', 'Temperature dependence']:
        raise ValueError('UniProt 响应列名异常')
    temperatures = {}
    for row in rows:
        if len(row) != 2 or not row[0] or row[0] in temperatures:
            raise ValueError('UniProt 响应出现异常行或重复登录号')
        temperatures[row[0]] = row[1]
    if not temperatures or not release:
        raise ValueError('UniProt 响应缺少条目或版本信息')
    if total_header is not None and len(temperatures) != int(total_header):
        raise ValueError('UniProt 响应的条目数与服务器声明不符')

    temperature_snapshot = {
        'source_url': temperature_url,
        'release': release,
        'retrieved_at_utc': datetime.now(timezone.utc).isoformat(),
        'response_sha256': sha256(payload).hexdigest(),
        'entry_count': len(temperatures),
        'annotations': temperatures,
    }
    with TemporaryDirectory(dir=diamond_dir, prefix='.uniprot_temperature_') as scratch:
        temp_cache = Path(scratch) / temperature_cache.name
        temp_cache.write_text(json.dumps(temperature_snapshot, ensure_ascii=False), encoding='utf-8')
        with suppress(FileExistsError):
            os.link(temp_cache, temperature_cache)

temperature_snapshot = json.loads(temperature_cache.read_text(encoding='utf-8'))
temperatures = temperature_snapshot['annotations']
if (temperature_snapshot['source_url'] != temperature_url
        or temperature_snapshot['entry_count'] != len(temperatures)):
    raise ValueError('缓存快照与查询配置不符')
print(f"UniProt 版本  {temperature_snapshot['release']}")
print(f"获取时间 UTC  {temperature_snapshot['retrieved_at_utc']}")
print(f'快照条目数    {len(temperatures):,}')
print(f"响应 SHA-256  {temperature_snapshot['response_sha256']}")

UniProt 版本  2026_04
获取时间 UTC  2026-11-11T08:24:24.047294+00:00
快照条目数    576,073
响应 SHA-256  d329da75223c7b9315ee8a39a5d53b078a43a6db9b1d958f9c6a1ba6b04fdee1


## 8. 为 DIAMOND 命中表追加温度注释

从 `sseqid`（`sp|accession|entry`）提取登录号，新增最后一列 `temperature_dependence`，保存到新的 TSV。原顺序及原有各列逐行保留。字段为空的条目输出空字符串。重复运行会逐行核验已有结果。

此列记录参考条目的温度注释。蛋白最适温度、温度稳定性和来源物种的最适生长温度，需依据注释内容分别解读。

In [9]:
temperature_hits_path = hits_path.with_name(hits_path.stem + '_temperature_dependence.tsv')
temperature_hit_columns = [*hit_columns, 'temperature_dependence']


def temperature_rows():
    with hits_path.open(encoding='utf-8', newline='') as handle:
        reader = csv.reader(handle, delimiter='\t', quoting=csv.QUOTE_NONE)
        if next(reader, None) != hit_columns:
            raise ValueError('DIAMOND 输入列名与配置不符')
        for row in reader:
            if len(row) != len(hit_columns):
                raise ValueError(f'DIAMOND 第 {reader.line_num} 行列数异常')
            parts = row[1].split('|')
            if len(parts) != 3 or parts[0] != 'sp':
                raise ValueError(f'Swiss-Prot 标识格式异常：{row[1]}')
            accession = parts[1]
            if accession not in temperatures:
                raise LookupError(f'当前 Swiss-Prot 快照中找不到 {accession}，需核实登录号历史')
            yield [*row, temperatures[accession]]


if not temperature_hits_path.exists():
    with TemporaryDirectory(dir=diamond_dir, prefix='.temperature_hits_') as scratch:
        temporary_output = Path(scratch) / temperature_hits_path.name
        with temporary_output.open('w', encoding='utf-8', newline='') as handle:
            writer = csv.writer(handle, delimiter='\t', lineterminator='\n')
            writer.writerow(temperature_hit_columns)
            writer.writerows(temperature_rows())
        with suppress(FileExistsError):
            os.link(temporary_output, temperature_hits_path)

temperature_row_count = 0
annotated_row_count = 0
annotated_subjects = set()
annotated_queries = set()
temperature_examples = []
with temperature_hits_path.open(encoding='utf-8', newline='') as handle:
    reader = csv.reader(handle, delimiter='\t')
    if next(reader, None) != temperature_hit_columns:
        raise ValueError('已有温度注释表的列名与配置不符')
    for saved, expected in zip_longest(reader, temperature_rows()):
        if saved != expected:
            raise ValueError('已有温度注释表与当前输入/快照不符，请指定新的输出文件名')
        temperature_row_count += 1
        if saved[-1]:
            annotated_row_count += 1
            annotated_subjects.add(saved[1])
            annotated_queries.add(saved[0])
            if len(temperature_examples) < 3 and saved[1] not in {
                example['sseqid'] for example in temperature_examples
            }:
                temperature_examples.append(dict(zip(temperature_hit_columns, saved)))
if temperature_row_count != hit_count:
    raise ValueError('注释后行数与 DIAMOND 命中记录数不符')

print(f'输出文件          {temperature_hits_path.relative_to(project)}')
print(f'总记录数          {temperature_row_count:,}')
print(f'有温度注释的记录  {annotated_row_count:,}')
print(f'对应参考条目数    {len(annotated_subjects):,}')
print(f'对应查询蛋白数    {len(annotated_queries):,}')
print(f'温度字段留空记录  {temperature_row_count - annotated_row_count:,}')
if temperature_examples:
    with pd.option_context('display.max_colwidth', 150):
        display(pd.DataFrame(temperature_examples)[['sseqid', 'temperature_dependence']])

输出文件          output/diamond/meer1000M_cd90_vs_swissprot_5fdf7a7839fe_temperature_dependence.tsv
总记录数          1,997,487
有温度注释的记录  53,952
对应参考条目数    1,914
对应查询蛋白数    35,233
温度字段留空记录  1,943,535


,sseqid,temperature_dependence
0,sp|P77908|FDHA_NEOTH,"BIOPHYSICOCHEMICAL PROPERTIES: Temperature dependence: Optimum temperature is 85 degrees Celsius (PubMed:4154039). In the absence of substrates, ..."
1,sp|P51691|ARS_PSEAE,BIOPHYSICOCHEMICAL PROPERTIES: Temperature dependence: Optimum temperature is 57 degrees Celsius. Incubation that exceeds 20 minutes above 50 deg...
2,sp|Q8R5M5|ACMSD_RAT,BIOPHYSICOCHEMICAL PROPERTIES: Temperature dependence: Optimum temperature is 40 degrees Celsius. {ECO:0000269|PubMed:11802786};


## 9. 按参考蛋白最适温度统计

按本次指定的严格边界 `<20°C`、`30–40°C（含端点）`、`>60°C` 统计。20°C 和 60°C 分别排除在第一组和第三组之外。读取明确的 optimum/optimal temperature 注释，并补充少数明确写为 maximum/highest activity 的条目。约数按注释标称值处理。温度范围和多条件的全部已提取最适温度均落入同一组时才纳入。跨区间、解析不明确及仅有稳定性描述的条目单列。

计数分别按参考登录号、命中记录和查询蛋白 ID 去重。查询蛋白可命中多组参考条目，因此组间查询数可能重叠。分组表示命中参考蛋白的温度证据。

In [10]:
from collections import Counter

temperature_groups = ['<20°C', '30–40°C', '>60°C']
optimum_anchor = re.compile(
    r'optim(?:um|al) temperatures?|temperatures? optim(?:um|a)|optimum activity at',
    re.IGNORECASE,
)
temperature_value = re.compile(
    r'(?P<low>\d+(?:\.\d+)?)'
    r'(?:\s*(?:-|to|and)\s*(?P<high>\d+(?:\.\d+)?))?'
    r'\s*degrees? Celsius',
    re.IGNORECASE,
)
# 原文明确用最高活性描述温度的条目。多底物条目保留全部温度。
maximum_activity_intervals = {
    'B2UPR7': [(30, 40)],
    'B2UQG6': [(45, 45)],
    'O05420': [(42, 42)],
    'P46541': [(40, 40)],
    'Q12382': [(30, 30)],
    'Q9R1E6': [(45, 55)],
    'W6JQJ6': [(65, float('inf')), (50, 50), (50, 60)],
}


def optimum_interval(text):
    clean = re.sub(r'\{[^}]+\}|\(PubMed:[^)]*\)', '', text)
    intervals = []
    for anchor in optimum_anchor.finditer(clean):
        tail = clean[anchor.end():]
        # 句号不截断小数。跳过“在最适温度下的稳定性”等回指。
        sentence = re.split(r'(?<!\d)\.(?!\d)|;', tail, maxsplit=1)[0]
        if not sentence.strip() or sentence.lstrip().startswith('with '):
            continue
        if 'depends on the study' in sentence:
            return None
        value = temperature_value.search(sentence)
        if value is None:
            continue
        low = float(value['low'])
        high = float(value['high'] or value['low'])
        prefix = sentence[:value.start()].lower()
        if re.search(r'below|less than|under', prefix):
            low = float('-inf')
        elif re.search(r'above|over|greater than|higher than|superior to', prefix):
            high = float('inf')
        intervals.append((low, high))
    if not intervals:
        return None
    return min(low for low, high in intervals), max(high for low, high in intervals)


def temperature_group(interval):
    if interval is None:
        return '缺少明确最适温度/需复核'
    low, high = interval
    if high < 20:
        return temperature_groups[0]
    if 30 <= low <= high <= 40:
        return temperature_groups[1]
    if low > 60:
        return temperature_groups[2]
    return '其他温度或跨区间'


# “above 60”给出了严格下界，可归入 >60。
# 逐条复核同句中有多个温度的注释，保留不同底物、盐浓度和 pH 条件的最适值。
# 这里的区间是这些明确最适值的包络，不包含稳定性和剩余活性测试温度。
reviewed_optimum_intervals = {
    'B8YG19': (49, 58), 'D4GYV5': (55, 75), 'E1ABX2': (55, 60),
    'G9MAP1': (37, 42), 'H6C4I7': (25, 50), 'I3R634': (60, 80),
    'M1GRN3': (45, 50), 'O52251': (0, 30), 'O57936': (55, 85),
    'O75003': (32.5, 37.5), 'P08910': (30, 45), 'P09773': (25, 42),
    'P0AEW6': (26, 39), 'P0CT50': (30, 40), 'P30602': (25, 50),
    'P80305': (50, 70), 'Q03649': (30, 45), 'Q12068': (30, 54),
    'Q12XX5': (50, 60), 'Q59226': (40, 50), 'Q5FA46': (45, float('inf')),
    'Q8DK23': (60, 70), 'Q8L0V4': (25, 30), 'Q8L3C7': (35, 50),
    'Q8NKS6': (70, 95), 'Q9AIK2': (50, 60), 'Q9HK01': (50, 63),
    'Q9LNJ9': (30, 45), 'Q9XDM9': (37, 42),
}


def classify_reference(accession, text):
    interval = optimum_interval(text)
    if accession in maximum_activity_intervals:
        values = maximum_activity_intervals[accession]
        interval = min(low for low, high in values), max(high for low, high in values)
    interval = reviewed_optimum_intervals.get(accession, interval)
    group = temperature_group(interval)
    if accession == 'Q9X9D5':
        interval, group = (60, float('inf')), '>60°C'
    return interval, group


# 针对阈值、跨区间及稳定性描述的回归检查。
assert temperature_group((20, 20)) == '其他温度或跨区间'
assert temperature_group((60, 60)) == '其他温度或跨区间'
assert temperature_group((10, 18)) == '<20°C'
assert temperature_group((35, 45)) == '其他温度或跨区间'
assert optimum_interval('Stable at 10 degrees Celsius for 20 minutes.') is None
assert classify_reference('O52251', temperatures['O52251'])[1] == '其他温度或跨区间'
assert classify_reference('I3R634', temperatures['I3R634'])[1] == '其他温度或跨区间'
assert classify_reference('Q9XDM9', temperatures['Q9XDM9'])[1] == '其他温度或跨区间'


In [11]:
reference_groups = {}
reference_intervals = {}
group_references = {group: set() for group in temperature_groups}
group_queries = {group: set() for group in temperature_groups}
group_hits = Counter()
reference_annotation_text = {}
unclassified_references = set()
for saved in temperature_rows():
    if not saved[-1]:
        continue
    accession = saved[1].split('|')[1]
    if accession not in reference_groups:
        interval, group = classify_reference(accession, saved[-1])
        reference_groups[accession] = group
        reference_intervals[accession] = interval
        reference_annotation_text[accession] = saved[-1]
    group = reference_groups[accession]
    if group in temperature_groups:
        group_references[group].add(accession)
        group_queries[group].add(saved[0])
        group_hits[group] += 1
    else:
        unclassified_references.add(accession)

temperature_summary = pd.DataFrame([
    {
        '参考蛋白最适温度': group,
        '参考条目数': len(group_references[group]),
        '命中记录数': group_hits[group],
        '查询蛋白数（组内去重）': len(group_queries[group]),
    }
    for group in temperature_groups
])
display(temperature_summary)
print('参考条目分组状态', dict(Counter(reference_groups.values())))
print('多组均有命中的查询数', sum(
    sum(query in group_queries[group] for group in temperature_groups) > 1
    for query in set().union(*group_queries.values())
))
cold_accessions = sorted(group_references['<20°C'])
print('低于 20°C 的参考条目', ', '.join(cold_accessions))

temperature_audit_path = hits_path.with_name(hits_path.stem + '_temperature_groups.tsv')
audit_columns = ['accession', 'optimum_temperature_interval_c', 'group', 'temperature_dependence']
audit_rows = [
    [accession, str(reference_intervals[accession]), reference_groups[accession],
     reference_annotation_text[accession]]
    for accession in sorted(reference_groups)
]
if temperature_audit_path.exists():
    with temperature_audit_path.open(encoding='utf-8', newline='') as handle:
        reader = csv.reader(handle, delimiter='\t')
        if next(reader, None) != audit_columns or list(reader) != audit_rows:
            raise ValueError('已有分组表与当前规则不符，请指定新输出路径')
else:
    with TemporaryDirectory(dir=diamond_dir, prefix='.temperature_groups_') as scratch:
        audit_temp = Path(scratch) / temperature_audit_path.name
        with audit_temp.open('w', encoding='utf-8', newline='') as handle:
            writer = csv.writer(handle, delimiter='\t', lineterminator='\n')
            writer.writerow(audit_columns)
            writer.writerows(audit_rows)
        with suppress(FileExistsError):
            os.link(audit_temp, temperature_audit_path)
print(f'参考条目分组核查表  {temperature_audit_path.relative_to(project)}')

,参考蛋白最适温度,参考条目数,命中记录数,查询蛋白数（组内去重）
0,<20°C,9,90,90
1,30–40°C,515,13002,11367
2,>60°C,411,14086,10014


参考条目分组状态 {'>60°C': 411, '其他温度或跨区间': 683, '30–40°C': 515, '缺少明确最适温度/需复核': 296, '<20°C': 9}
多组均有命中的查询数 1275
低于 20°C 的参考条目 A0A1V0ELS9, A0A3G9JYH7, A0FKE6, A8PUY5, J7H670, O52631, P81650, Q6PRW6, Q9LQF2
参考条目分组核查表  output/diamond/meer1000M_cd90_vs_swissprot_5fdf7a7839fe_temperature_groups.tsv
